In [1]:
!python -m pip install --upgrade pip
!pip install "numpy<2"
!pip -q install -U sentence-transformers
!pip -q install pyvi
!pip install unidecode
!pip -q install -q setuptools
!pip -q install easyocr
!pip install -q vietocr 
!pip -q install scandir
!pip -q install usearch
!pip install ipywidgets --upgrade


import warnings
warnings.filterwarnings('ignore')

# Install detectron2, suppressing stderr
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git' 2> /dev/null

# Clone repository and install DeepSolo++, suppressing stderr
!git clone https://github.com/TinhAnhGitHub/DeepSolo 2> /dev/null
%cd DeepSolo/DeepSolo++
!pip install -r requirements.txt 2> /dev/null
!python setup.py build develop 2> /dev/null


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 0.22.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.10.0, but you have google-cloud-bigquery 2.34.4 which is incompatible.
bigframes 0.22.0 requires google-cloud-storage>=2.0.0, but you have google-cloud-storage 1.44.0 which is incompatible.
bigframes 0.22.0 requires pandas<2.1.4,>=1.5.0, but you have pandas 2.2.2 which is incompatible.
dataproc-jupyter-plugin 0.1.79 requires pydantic~=1.10.0, but you have pydantic 2.8.2 which is incompatible.
pointpats 2.5.0 requires shapely>=2, but you have shapely 1.8.5.post1 which is incompatible.
spaghetti 1.7.6 requires shapely>=2.0.1, but you have

In [2]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from scene_text_detection import SceneTextDetection
from pyvi.ViTokenizer import tokenize
from tqdm import tqdm
from typing import List, Dict, Tuple, Any
from collections import OrderedDict
from PIL import Image
import matplotlib.pyplot as plt
import scandir
from collections import OrderedDict
import cProfile
from sklearn.decomposition import PCA
import pstats
import io
from contextlib import redirect_stdout

In [3]:
def calculate_angle(
    points: np.ndarray
) -> float:
    """Calculate the angle of a line formed by a set of points

    Args:
        points (np.ndarray): Array of control points, shape = (50, )

    Returns:
        float: angle in degrees
    """
    points = points.reshape(-1, 2)
    
    pca = PCA(n_components=2, random_state=42)
    pca.fit(points)
    angle = np.arctan2(pca.components_[0, 1], pca.components_[0, 0])
    return np.degrees(angle)


def get_rect_points(rect: Tuple[Tuple[float, float], Tuple[float, float], float]) -> np.ndarray:
    box = cv2.boxPoints(rect)
    return np.intp(box)

def calculate_center_angle(point1: Tuple[float, float], point2: Tuple[float, float]) -> float:
    dx = point2[0] - point1[0]
    dy = point2[1] - point1[1]
    return np.degrees(np.arctan2(dy, dx))

def distance_between_rects(rect1: np.ndarray, rect2: np.ndarray) -> float:
    return np.min(np.linalg.norm(rect1[:, np.newaxis] - rect2, axis=2))

def collate_fn(batch: List[Tuple[np.ndarray, str]]) -> Tuple[List[str]]:
    """Custom collate function for DataLoader."""
    if len(batch) == 1:
        return batch
    paths = zip(*batch)
    return list(paths)
    

def min_area_rectangle(bounding_points: np.ndarray) -> Tuple[Tuple[float, float], Tuple[float, float], float]:
    """Turning bounding points into rectangle ( minimum area )

    Args:
        bounding_points (np.ndarray): array shape (n_instance_point, 4) -> (xmin, ymin, xmax, ymax)

    Returns:
        Tuple[Tuple[float, float], Tuple[float, float], float]: return 
        ((center_x, center_y), (width, height), angle of rotation)
    """
    bounding_points = np.hsplit(bounding_points, 2)
    bounding_points = np.vstack([bounding_points[0], bounding_points[1][::-1]])

    rect = cv2.minAreaRect(bounding_points.astype(np.float32))
    return rect
    

def detect_text_lines(
    rect_list: List[Tuple[Tuple[float, float], Tuple[float, float], float]],
    control_pts_list: List[np.ndarray],
    distance_threshold: float = 20,
    angle_threshold: float = 7,
    height_ratio_threshold: float = 0.2,
    center_angle_threshold: float = 10
) -> List[List[int]]:

    sentences = []
    remaining_indices = list(range(len(rect_list)))

    while remaining_indices:
        sorted_indices = sorted(remaining_indices, key=lambda i: rect_list[i][0][0])

        sample_rect_idx = sorted_indices.pop(0)
        remaining_indices.remove(sample_rect_idx)
        current_sentence = [sample_rect_idx]

        sample_rect = get_rect_points(rect_list[sample_rect_idx])
        sample_height = rect_list[sample_rect_idx][1][1]
        sample_angle = calculate_angle(control_pts_list[sample_rect_idx])
        sample_center = rect_list[sample_rect_idx][0]

        while sorted_indices:
            min_distance = float('inf')
            next_rect_idx = None

            for idx in sorted_indices:
                curr_rect = get_rect_points(rect_list[idx])
                curr_height = rect_list[idx][1][1]
                curr_angle = calculate_angle(control_pts_list[idx])

                distance = distance_between_rects(sample_rect, curr_rect)
                height_ratio = min(sample_height, curr_height) / max(sample_height, curr_height)
                angle_diff = abs(sample_angle - curr_angle)
                curr_center = rect_list[idx][0]

                if (distance < distance_threshold and
                    height_ratio > height_ratio_threshold and
                    angle_diff < angle_threshold):
                    avg_angle = (sample_angle + curr_angle) / 2
                    center_angle = calculate_center_angle(sample_center, curr_center)
                    center_angle_diff = abs(center_angle - avg_angle)

                    if center_angle_diff < center_angle_threshold and distance < min_distance:
                        min_distance = distance
                        next_rect_idx = idx

            if next_rect_idx is None:
                break

            current_sentence.append(next_rect_idx)
            sorted_indices.remove(next_rect_idx)
            remaining_indices.remove(next_rect_idx)

            sample_rect = get_rect_points(rect_list[next_rect_idx])
            sample_height = rect_list[next_rect_idx][1][1]
            sample_angle = calculate_angle(control_pts_list[next_rect_idx])
            sample_center = rect_list[next_rect_idx][0] 

        sentences.append(current_sentence)

    return sentences

In [4]:
class SceneTextDetector:
    """
    Class for scene text detection using DeepSolo++
    """
    def __init__(self, detector_model_path: str, detector_config_path: str) -> None:
        self.detector = SceneTextDetection(
            model_weight=detector_model_path,
            config_file=detector_config_path
        )
    def detect(self, image_paths: List[str]) -> List[Dict[str, Any]]:
        
        res = self.detector.process_images(image_paths)
          
        return res

In [5]:
import os
import json
import string
import unidecode
from collections import OrderedDict
from tqdm import tqdm
from typing import List

valid_chars = string.ascii_uppercase + string.digits

class MainProcessor:
    """
    Main class for processing images and extracting text information.
    """
    def __init__(self, root_dir: str, batch_size: int, output_folder: str, confidence_threshold: float, checkpoint_interval: int = 100):
        self.root_dir = root_dir
        self.batch_size = batch_size
        self.output_folder = output_folder
        self.confidence_threshold = confidence_threshold
        
        self.text_detector = SceneTextDetector(
            detector_model_path='/kaggle/input/deepsolo-plus-plus-weight/r50_data3_multilingual_finetune.pth',
            detector_config_path='./configs/R_50/mlt19_multihead/finetune.yaml'
        )
        
        self.global_index2image = OrderedDict()
        self.image_path2ocr_output = {}
        self.image_path2ocr_output_path  = os.path.join(output_folder, 'image_path2ocr_output_path.json')
        
        self.checkpoint_interval = checkpoint_interval
        self.checkpoint_file = os.path.join(output_folder, 'checkpoint.json')

    def load_checkpoint(self):
        if os.path.exists(self.checkpoint_file):
            with open(self.checkpoint_file, 'r') as f:
                checkpoint_data = json.load(f)
            
            self.global_index2image = OrderedDict(checkpoint_data['global_index2image'])
            global_index = max(map(int, self.global_index2image.keys())) + 1 if self.global_index2image else 0
            if os.path.exists(self.image_path2ocr_output_path):
                with open(self.image_path2ocr_output_path, 'r', encoding='utf-8') as f:
                    self.image_path2ocr_output = json.load(f)
            else:
                self.image_path2ocr_output = {}
            print(f"Load from checkpoint: {global_index}, continue from {checkpoint_data['processed_images'][-1]}")
            return global_index, checkpoint_data['processed_images']
        return 0, []
    
    def save_checkpoint(self, global_index: int, processed_images: List[str]):
        checkpoint_data = {
            'global_index2image': self.global_index2image,
            'processed_images': processed_images
        }
        
        with open(self.checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=4)
        with open(self.image_path2ocr_output_path, 'w', encoding='utf-8') as f:
            json.dump(self.image_path2ocr_output, f, indent=4, ensure_ascii=False)
            
        print("Checkpoint saved!!!")
    
    def get_image_paths(self) -> List[str]:
        if self.root_dir.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp')):
            return [self.root_dir]
        image_paths = []
        for root, _, files in os.walk(self.root_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp')):
                    image_paths.append(os.path.join(root, file))
        
        if not image_paths:
            print(f"No image found in this directory: {self.root_dir}")
            return []
        return sorted(image_paths)

    def is_valid_image(self, image_path: str) -> bool:
        """Kiểm tra xem ảnh có mở được không."""
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Cannot read image: {image_path}")
                return False
            return True
        except Exception as e:
            print(f"Error reading image {image_path}: {str(e)}")
            return False

    def process_images(self):
        """
        Process images and extract information
        """
        image_paths = self.get_image_paths() 
        if len(image_paths) == 0:
            return self.global_index2image, self.image_path2ocr_output
            
        global_index, processed_images = self.load_checkpoint()
        print("Reading Images")
        image_paths = [path for path in image_paths if path not in processed_images]
        image_paths.sort()
        
        print("Start processing...")

        for batch_start in tqdm(range(0, len(image_paths), self.batch_size), desc="Processing Batches"):
            batch_paths = image_paths[batch_start:batch_start + self.batch_size]
            # Kiểm tra ảnh hợp lệ trước khi xử lý
            batch_paths = [path for path in batch_paths if self.is_valid_image(path)]
            if not batch_paths:
                print("No valid images in the current batch. Skipping...")
                continue

            try:
                detection_results = self.text_detector.detect(batch_paths)
            except Exception as e:
                print(f"Error during detection: {str(e)}")
                continue  # Bỏ qua batch này và tiếp tục

            if not detection_results or not isinstance(detection_results, list):
                print(f"Invalid detection results for batch: {batch_paths}")
                continue
            
            for i, result in enumerate(detection_results):
                image_path = batch_paths[i]
                print(f"Processing image: {image_path}")
                if not result or not isinstance(result, list) or len(result) == 0:
                    print(f"Invalid or empty result for image: {image_path}")
                    continue

                if 'instances' not in result[0]:
                    print(f"No instances found in image: {image_path}")
                    continue

                instances = result[0]['instances']
                
                if not instances:
                    print(f"No text detected in image: {image_path}")
                    continue
                
                try:
                    bds = instances.bd.cpu().detach().numpy() 
                    ctrl_pts = instances.ctrl_points.cpu().detach().numpy()     
                    recs = instances.recs.cpu().detach().numpy()
                    decoded_recs = []
                    for j, language_id in enumerate(result[0]['instances'].languages.cpu().numpy()):
                        language = self.text_detector.detector.language_list[int(language_id)]
                        decoded_rec = self.text_detector.detector.ctc_decode_recognition(recs[j], language)
                        decoded_recs.append(decoded_rec)
                    
                    rect_list = [min_area_rectangle(bd) for bd in bds] 
                    sentences = detect_text_lines(rect_list, ctrl_pts)
                    word_texts = []
                    
                    for idx, sentence in enumerate(sentences):
                        words_in_sentence = []
                        for word_idx in sentence:
                            if word_idx < len(decoded_recs):
                                words_in_sentence.append(unidecode.unidecode(decoded_recs[word_idx]))
                        sentence_text = ' '.join(words_in_sentence)
                        if sentence_text:
                            word_texts.append(sentence_text)
                            
                    if word_texts:
                        self.global_index2image[str(global_index)] = image_path
                        if image_path not in self.image_path2ocr_output:
                            self.image_path2ocr_output[image_path] = []
                    
                        self.image_path2ocr_output[image_path].extend(word_texts)
                        global_index += 1
                except Exception as e:
                    print(f"Error processing image {image_path}: {str(e)}")

            processed_images.extend(batch_paths)
            if len(processed_images) % self.checkpoint_interval == 0:
                self.save_checkpoint(global_index, processed_images)

        return self.global_index2image, self.image_path2ocr_output

batch_size = 10
output_folder = '/kaggle/working'
os.makedirs(output_folder, exist_ok=True)
confidence_threshold = 0.85
video_root_dir = '/kaggle/input/new-index-final/keyframes'

for video_folder in os.listdir(video_root_dir):
    if not (video_folder.startswith("L10") or video_folder.startswith("L11")):
        continue  # Bỏ qua thư mục không khớp

    video_path = os.path.join(video_root_dir, video_folder)
    if not os.path.isdir(video_path):
        print(f"{video_path} is not a directory. Skipping...")
        continue

    # Khởi tạo MainProcessor
    main_process = MainProcessor(
        root_dir=video_path,  
        batch_size=batch_size,
        output_folder=output_folder,
        confidence_threshold=confidence_threshold
    )

    global2img, image_path2ocr_output = main_process.process_images()

    output_file = f'/kaggle/working/{video_folder}.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(image_path2ocr_output, f, indent=4, ensure_ascii=False)


[10/12 17:48:11 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/deepsolo-plus-plus-weight/r50_data3_multilingual_finetune.pth ...
[10/12 17:48:15 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/deepsolo-plus-plus-weight/r50_data3_multilingual_finetune.pth ...
Reading Images
Start processing...


Processing Batches:   0%|          | 0/45 [00:00<?, ?it/s]

[10/12 17:48:22 detectron2]: Detected instances in 0.68s


Processing Batches:   2%|▏         | 1/45 [00:07<05:15,  7.18s/it]

Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0001.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0002.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0003.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0004.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0005.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0006.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0007.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0008.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0009.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0010.jpg
[10/12 17:48:27 detectron2]: Detected instances in 0.44s
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0011.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L10_V016/0012.jpg
Processing image: /kaggle/input

...

Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0621.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0622.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0623.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0624.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0625.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0626.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0627.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0628.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0629.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0630.jpg
[10/12 23:36:20 detectron2]: Detected instances in 0.50s


Processing Batches:  98%|█████████▊| 64/65 [07:55<00:05,  5.32s/it]

Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0631.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0632.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0633.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0634.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0635.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0636.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0637.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0638.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0639.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0640.jpg
[10/12 23:36:21 detectron2]: Detected instances in 0.48s


Processing Batches: 100%|██████████| 65/65 [07:56<00:00,  7.34s/it]

Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0641.jpg
Processing image: /kaggle/input/new-index-final/keyframes/L11_V002/0642.jpg
